# MiniVLMDocEval — Colab runner (terminal)

This is the **only** notebook. It is a thin terminal: pull the latest code from GitHub and run a `.py` script on the GPU. All real logic lives in `.py` files, edited locally (by Claude) and synced via git.

**Loop:** Claude edits scripts locally → push → you re-run the *Run* cell (`git pull && python ...`).

One-time per session: run the **GPU+clone**, **Drive mount**, and **setup** cells. All eval outputs (predictions + summary tables) are written to Google Drive (`WORK_DIR`) so they persist across sessions and `--reuse` can resume.

Set runtime to **T4 GPU** before running.

In [ ]:
# --- One-time per session: GPU + clone ---
import os
REPO_URL = 'https://github.com/SajjadPSavoji/MiniVLMDocEval.git'
REPO = '/content/MiniVLMDocEval'

!nvidia-smi -L || echo 'NO GPU — Runtime > Change runtime type > T4 GPU'
if not os.path.isdir(REPO):
    assert REPO_URL, 'Set REPO_URL'
    !git clone $REPO_URL $REPO
%cd $REPO

In [ ]:
# --- One-time per session: mount Google Drive for persistent outputs ---
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/MiniVLMDocEval/outputs'
os.makedirs(WORK_DIR, exist_ok=True)
print('outputs (predictions + summary tables) ->', WORK_DIR)

In [ ]:
# --- Run cell: full eval -> Drive. Re-run after each Claude push to iterate.
#     --reuse resumes automatically if a session dropped mid-run.
#     First time, validate small by adding `--data OCRBench` (3 models x 1k).
#     Drop it for the full 5-benchmark suite.
!git pull --quiet && python scripts/run_eval.py --work-dir $WORK_DIR

In [ ]:
# --- Run cell: the tight loop. Edit the script + args; re-run to iterate. ---
!git pull --quiet && python scripts/smoke_test.py --model SmolVLM2-500M --data OCRBench --n 5 --score